# 2.5 — does focal loss add anything on top of rotation?

Six runs with free-angle rotation throughout. Only the model and the imbalance handling
vary.

| config | model | loss | gamma | sampler |
|---|---|---|---|---|
| `control` | `baseline_v2` | cross-entropy | — | inverse-sqrt |
| `control_v1` | `baseline_cnn` | cross-entropy | — | inverse-sqrt |
| `focal_g2` | `baseline_v2` | focal | 2.0 | none |
| `focal_g1` | `baseline_v2` | focal | 1.0 | none |
| `focal_g2_sampler` | `baseline_v2` | focal | 2.0 | inverse-sqrt |
| `focal_g1_sampler` | `baseline_v2` | focal | 1.0 | inverse-sqrt |

## What the first attempt showed, and why it was rerun

The first pass gave `control` 0.7830 against phase 2's 0.8800, and the cause was a bug, not
a result: moving these configs into a subdirectory broke `defaults.yaml` inheritance. The
loader looked for it in the immediate parent only, found none, and merged nothing —
**silently**. All five arms trained with batch 512 instead of 256, learning rate 1e-3
instead of 7e-4, 40 epochs instead of 50 and patience 5 instead of 7.

The symptom was visible in the table: every arm's best epoch was 36-39 out of 40, so none of
them had converged. The loader now walks upward for `defaults.yaml` and warns loudly when
there is none.

`control_v1` is new and settles the other confound. `baseline_v2` had **no prior runs** —
it was built to fix overfitting that rotation turned out to fix on its own, so cutting the
model from 157k to 34k parameters may simply have cost capacity. `control_v1` is the
phase-2 arm that scored 0.8800, so it also checks that the retuned defaults and the GPU
rotation path reproduce it.

## What still holds from the first pass

The ordering was internally consistent, since all arms shared the same wrong settings. Focal
**with** the sampler tied the control (0.7828 vs 0.7830); focal **without** it was clearly
worse (-0.058, -0.063). That says the sampler does the work and focal does not replace it,
which matches the mechanism: the sampler makes rare classes common in a batch, and focal
then down-weights exactly those examples once the model gets them right.

## How to judge

Compare against `control`, and treat a gain under ~0.02 as unconfirmed — we are deep enough
into validation selection that roughly that much optimism is already priced in. `Near-full`
has 30 validation wafers, so point estimates alone are not a ranking.

## 1. Setup

Same bootstrap as the phase-2 notebook; every step is a no-op if already done.

In [1]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

gpu     NVIDIA L4
drive   mounted
dataset 1.88 GiB


## 2. W&B

`wandb login` in the terminal is the reliable path — Colab Secrets time out when the runtime
is driven from VS Code. `~/.netrc` is read by both this kernel and any terminal process.

In [2]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- run `wandb login` in the terminal, then rerun")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")

  wandb ready, project 'wm811k-wafer-defects'


## 3. Run the five arms

Sequentially, on one loaded copy of the source table. `transform_device=cuda` is on: rotation
becomes a single batched `grid_sample` and the encoding happens after the transfer, which is
worth the most on precisely this augmentation.

Each arm writes its own artifacts and appends to the results list as it finishes, so an
interrupted session keeps whatever completed.

In [3]:
import time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.training.runner import run_experiment

CONFIGS = sorted((REPO / "configs/train/v25_focal").glob("*.yaml"))
OVERRIDES = ["data.transform_device=cuda"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}"]

dataframe = load_wm811k_dataframe(DATASET)
results = []

for path in CONFIGS:
    config = load_experiment_config(path, overrides=OVERRIDES)
    # Guard against the bug that invalidated the first pass: these values come
    # from configs/train/defaults.yaml, so if inheritance breaks again the run
    # stops here instead of quietly training on dataclass defaults.
    assert config.trainer.max_epochs == 50, "defaults.yaml was not inherited"
    assert config.trainer.batch_size == 256, "defaults.yaml was not inherited"

    print(f"\n=== {config.name}  ({config.model.name}, {config.imbalance.loss}, "
          f"gamma {config.imbalance.focal_gamma}, sampling {config.imbalance.sampling})")
    started = time.monotonic()
    result = run_experiment(config, overwrite=True, dataframe=dataframe)
    macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
    results.append({
        "run": config.name,
        "model": config.model.name,
        "loss": config.imbalance.loss,
        "gamma": config.imbalance.focal_gamma if config.imbalance.loss == "focal" else None,
        "sampler": config.imbalance.sampling,
        "macro_f1": round(float(macro.point_estimate), 4),
        "ci_lower": round(float(macro.ci_lower), 4),
        "ci_upper": round(float(macro.ci_upper), 4),
        "best_epoch": result.fit.best_epoch,
        "epochs": len(result.fit.history),
        "minutes": round((time.monotonic() - started) / 60, 1),
    })
    row = results[-1]
    flag = "  <-- still improving at the cap" if row["best_epoch"] >= row["epochs"] - 2 else ""
    print(f"    macro-F1 {macro.point_estimate:.4f} "
          f"[{macro.ci_lower:.4f}, {macro.ci_upper:.4f}]  "
          f"best {row['best_epoch']}/{row['epochs']}  {row['minutes']:.1f} min{flag}")


=== baseline_v2-rotation-sampler  (baseline_v2, cross_entropy, gamma 2.0, sampling weighted)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: vlad-yelisieiev-bicocca (vlad-yelisieiev-bicocca-milano-bicocca) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
epoch_seconds,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▄▄▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████████
train_loss,█▅▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_samples,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation_accuracy,▁▄▅▅▅▆▆▆▆▇▇▆▇▇▇▇▇▇▇██▇▇▇▇▇▇█▇▇▇█████████
validation_balanced_accuracy,▁▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▆▇▇▇▇▇▇▇▇████▇███████
validation_f1_Center,▁▂▂▃▃▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇██▇▇██████████████
validation_f1_Donut,▁▁▂▂▂▃▄▃▃▃▄▅▄▄▄▅▅▆▆▅▅▇▇▅▆▆▆▇▆▇█▇▇▇▇███▇█
+10,...


    macro-F1 0.8061 [0.7908, 0.8194]  best 48/50  3.8 min  <-- still improving at the cap

=== baseline_cnn-rotation-sampler  (baseline_cnn, cross_entropy, gamma 2.0, sampling weighted)


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
epoch_seconds,█▁▁▁▂▁▁▁▁▁▂▂▁▁▂▂▁▁▁▂▁▁▁▂▁
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▄▅▅▆▆▆▇▇▇▇▇▇▇███████████
train_loss,█▅▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
train_samples,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation_accuracy,█████▁███████████████████
validation_balanced_accuracy,▁▁▄▆▅▂▇▆▆▇▇▆▆▇▆▇▇▇▆▇▇█▇██
validation_f1_Center,▅▆▅▄▆▄▆▄▇▁▇▇▆▆▇▅▅██▇▆▆▇▂▂
validation_f1_Donut,▄▃▇▆▃▂▇█▇▆█▇▇▇█▇▇▇▇▆▅▃▇▆▁
+10,...


...

    macro-F1 0.8084 [0.7912, 0.8223]  best 49/50  3.8 min  <-- still improving at the cap

=== baseline_v2-rotation-focal_g2  (baseline_v2, focal, gamma 2.0, sampling shuffle)


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
epoch_seconds,█▂▂▂▂▁▁▂▁▁▁▂▁▁▂▁▁▂▁▁▂▁▁▁▂▂
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▅▅▆▆▆▆▇▇▇▇▇▇▇▇███████████
train_loss,█▄▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
train_samples,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation_accuracy,▁▃▃▄▄▅▅▆▆▆▇▆▇▇▇▇▇█████████
validation_balanced_accuracy,▁▂▄▅▅▅▅▆▆▆▆▇▆▇▇▇▇▇██▇▇▇███
validation_f1_Center,▁▃▄▅▅▅▅▅▆▆▆▇▇▆▇▇▇▇█████▇██
validation_f1_Donut,▁▁▁▁▁▁▃▅▂▄▆▇▆▇▇▅▇▇█▇█▇█▇██
+10,...


    macro-F1 0.7024 [0.6847, 0.7168]  best 19/26  2.4 min

=== baseline_v2-rotation-focal_g2-sampler  (baseline_v2, focal, gamma 2.0, sampling weighted)


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
epoch_seconds,█▁▁▁▂▁▂▂▁▂▁▂▂▂▁▁▁▁▂▂▂▂▁▁▂▁▁▂▁▁▂▂▁▂▁▂▂▁▂▁
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████████████
train_loss,█▇▆▆▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
train_samples,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation_accuracy,▁▃▄▅▅▆▆▆▆▇▇▇█▇▇▇▇▆▇█▆▇▆▇▇▆▆█▆▆▆▇▆▇▇██▇▇▇
validation_balanced_accuracy,▁▂▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▆▇▆▇▇▇▇▇▇█▇██▇██▇████
validation_f1_Center,▁▃▃▄▅▆▆▇▇▇██▇▇▇████▇█▇██▇▇██████████████
validation_f1_Donut,▁▁▂▁▄▄▃▂▅▅▆▅▆▇▄▆▅▆▆▆▇█▆▅▃▄▆▆▇▆▆▅▇▆▆██▇▆▇
+10,...


    macro-F1 0.7995 [0.7829, 0.8133]  best 45/50  3.9 min


## 4. Read the result

`clears_control` is the test that matters: an arm counts only if its interval sits entirely
above the control's point estimate. With nine classes and 30 `Near-full` wafers in
validation, point estimates alone are not a ranking.

In [4]:
frame = pd.DataFrame(results)
control = frame.loc[frame["run"] == "baseline_v2-rotation-sampler", "macro_f1"].squeeze()
frame["vs_control"] = (frame["macro_f1"] - control).round(4)
frame["clears_control"] = frame["ci_lower"] > control
# A best epoch at the cap means the run was still improving when it stopped:
# that number is a floor, not a result.
frame["truncated"] = frame["best_epoch"] >= frame["epochs"] - 2

pd.set_option("display.width", 220)
display(frame.sort_values("macro_f1", ascending=False))

if frame["truncated"].any():
    print("\nTruncated (still improving at the cap):",
          ", ".join(frame.loc[frame["truncated"], "run"]))
    print("Raise trainer.max_epochs before drawing conclusions from those.")

v1 = frame.loc[frame["run"] == "baseline_cnn-rotation-sampler", "macro_f1"]
if not v1.empty:
    print(f"\nModel comparison, rotation and sampler held fixed:")
    print(f"  baseline_cnn {float(v1.squeeze()):.4f}   baseline_v2 {float(control):.4f}")
    print("  Phase 2 scored baseline_cnn + rotation at 0.8800 with patience 10.")

winners = frame[frame["clears_control"] & ~frame["run"].str.endswith("rotation-sampler")]
if winners.empty:
    print("\nNothing clears the control. Imbalance handling is not the binding "
          "constraint here either -- report that as the finding.")
else:
    print("\nClears the control:", ", ".join(winners["run"]))
    print("Treat gains under ~0.02 as unconfirmed until the test split is opened.")

,run,model,loss,gamma,sampler,macro_f1,ci_lower,ci_upper,best_epoch,epochs,minutes,vs_control,clears_control,truncated
1,baseline_cnn-rotation-sampler,baseline_cnn,cross_entropy,NaN,weighted,0.8696,0.8538,0.8822,18,25,2.2,0.0635,True,False
3,baseline_v2-rotation-focal_g1-sampler,baseline_v2,focal,1.0,weighted,0.8084,0.7912,0.8223,49,50,3.8,0.0023,False,True
0,baseline_v2-rotation-sampler,baseline_v2,cross_entropy,NaN,weighted,0.8061,0.7908,0.8194,48,50,3.8,0.0000,False,True
5,baseline_v2-rotation-focal_g2-sampler,baseline_v2,focal,2.0,weighted,0.7995,0.7829,0.8133,45,50,3.9,-0.0066,False,False
4,baseline_v2-rotation-focal_g2,baseline_v2,focal,2.0,shuffle,0.7024,0.6847,0.7168,19,26,2.4,-0.1037,False,False
2,baseline_v2-rotation-focal_g1,baseline_v2,focal,1.0,shuffle,0.7005,0.6844,0.7144,19,26,2.3,-0.1056,False,False



Truncated (still improving at the cap): baseline_v2-rotation-sampler, baseline_v2-rotation-focal_g1-sampler
Raise trainer.max_epochs before drawing conclusions from those.

Model comparison, rotation and sampler held fixed:
  baseline_cnn 0.8696   baseline_v2 0.8061
  Phase 2 scored baseline_cnn + rotation at 0.8800 with patience 10.

Nothing clears the control. Imbalance handling is not the binding constraint here either -- report that as the finding.


In [5]:
OUTPUT = REPO / "output/v25_focal"
OUTPUT.mkdir(parents=True, exist_ok=True)
frame.to_csv(OUTPUT / "results.csv", index=False)
if HAS_DRIVE:
    # Local disk dies with the session; Drive does not.
    shutil.copy2(OUTPUT / "results.csv", DRIVE / "v25_focal_results.csv")
    print("copied to", DRIVE / "v25_focal_results.csv")
print(OUTPUT / "results.csv")

copied to /content/drive/MyDrive/BICOCCA/FDL/v25_focal_results.csv
/content/fdl-project/output/v25_focal/results.csv


## 5. What follows

If an arm clears the control, it becomes part of the pipeline and the next series runs on
top of it. If nothing does — the likelier outcome given that every imbalance arm in phase 2
landed inside ±0.013 — then the finding is that **augmentation is the only lever that
mattered on this dataset**, and the report says so with the numbers behind it.

Either way the next step is the same: three seeds on whichever setup wins, to separate a
real difference from selection on a noisy validation metric.